In [ ]:
#парсит в 3 файла image_text_pairs2_1.csv , annotations_test.csv и annotations_train.csv
import json
import csv
import os

# Пути к файлам
base_dir = "dataset2_1"
file_configs = [
    {"json_path": os.path.join(base_dir, "annotations_val.json"), "csv_path": "image_text_pairs2_1.csv"},
    {"json_path": os.path.join(base_dir, "annotations_test.json"), "csv_path": "annotations_test.csv"},
    {"json_path": os.path.join(base_dir, "annotations_train.json"), "csv_path": "annotations_train.csv"}
]

# Функция для вычисления bbox из segmentation
def compute_bbox_from_segmentation(segmentation):
    if not segmentation or not isinstance(segmentation, list):
        return []
    # Предполагаем, что segmentation содержит список [x1, y1, x2, y2, ...]
    flat_coords = [coord for poly in segmentation for coord in poly]
    if len(flat_coords) < 4:
        return []
    x_coords = flat_coords[0::2]  # Все x-координаты
    y_coords = flat_coords[1::2]  # Все y-координаты
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    return [x_min, y_min, x_max - x_min, y_max - y_min]

# Функция для обработки одного JSON-файла и создания CSV
def process_annotations(json_path, csv_path):
    # Проверка существования JSON-файла
    if not os.path.exists(json_path):
        print(f"Файл {json_path} не найден, пропускаем.")
        return 0
    
    # Загрузка JSON
    with open(json_path, "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    # Словарь для хранения данных: {image_id: [{"text": str, "bbox": list}]}
    image_data = {}

    # Проходим по всем аннотациям
    for ann in coco_data.get("annotations", []):
        image_id = ann["image_id"]
        
        # Извлекаем текст
        text = (ann.get("text", "") or 
                ann.get("attributes", {}).get("translation", "") or
                ann.get("caption", ""))
        
        # Извлекаем bbox
        bbox = ann.get("bbox", [])
        if not isinstance(bbox, list) or len(bbox) != 4:
            # Если bbox отсутствует, пробуем вычислить из segmentation
            bbox = compute_bbox_from_segmentation(ann.get("segmentation", []))
        
        if image_id not in image_data:
            image_data[image_id] = []
        
        if text or bbox:
            image_data[image_id].append({"text": text, "bbox": bbox})

    # Создаём список записей для CSV
    image_records = []

    for image in coco_data.get("images", []):
        image_id = image["id"]
        image_file = os.path.join(base_dir, "images", image["file_name"])
        
        if image_id in image_data:
            # Для каждой аннотации создаем отдельную запись
            for data in image_data[image_id]:
                text = data["text"]
                bbox = data["bbox"]
                bbox_str = ",".join(map(str, bbox)) if bbox else ""
                image_records.append((image_file, text, bbox_str))
        else:
            # Добавляем запись для изображения без аннотаций
            image_records.append((image_file, "", ""))

    # Сохраняем в CSV
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image_path", "text", "bbox"])  # Заголовок
        writer.writerows(image_records)

    print(f"Обработано {len(image_records)} записей для {json_path}")
    print(f"Результаты сохранены в {csv_path}")
    if image_records:
        print(f"Пример записи: {image_records[0]}")
    else:
        print("Нет данных")
    
    return len(image_records)

# Обработка всех файлов
total_records = 0
for config in file_configs:
    total_records += process_annotations(config["json_path"], config["csv_path"])

print(f"\nОбщее количество обработанных записей: {total_records}")

Обработано 34623 записей для dataset2_1/annotations_val.json
Результаты сохранены в image_text_pairs2_1.csv
Пример записи: ('dataset2_1/images/2638.jpg', 'Своими', '891.09,1779.73,276.26999999999987,94.28999999999996')
Обработано 34670 записей для dataset2_1/annotations_test.json
Результаты сохранены в annotations_test.csv
Пример записи: ('dataset2_1/images/2013.jpg', 'узнавая', '3028.14,430.23,328.6800000000003,62.079999999999984')
Обработано 335962 записей для dataset2_1/annotations_train.json
Результаты сохранены в annotations_train.csv
Пример записи: ('dataset2_1/images/0_0.jpg', 'прил.,', '1301.01,1430.32,163.73000000000002,68.93000000000006')

Общее количество обработанных записей: 405255


NameError: name 'dataframes' is not defined

In [2]:
import pandas as pd
import os

# Пути к CSV файлам
base_dir = "dataset2_1"
csv_files = [
    {"csv_path": "image_text_pairs2_1.csv", "df_name": "val_df"},
    {"csv_path": "annotations_test.csv", "df_name": "test_df"},
    {"csv_path": "annotations_train.csv", "df_name": "train_df"}
]

# Список для хранения DataFrame'ов
dataframes = {}

# Обработка каждого CSV файла
for config in csv_files:
    csv_path = config["csv_path"]
    df_name = config["df_name"]
    
    # Проверка существования файла
    if not os.path.exists(csv_path):
        print(f"Файл {csv_path} не найден, пропускаем.")
        continue
    
    # Загрузка CSV в DataFrame
    df = pd.read_csv(csv_path, encoding="utf-8")
    
    # Проверка наличия ожидаемых столбцов
    expected_columns = ["image_path", "text", "bbox"]
    if not all(col in df.columns for col in expected_columns):
        print(f"Ошибка: файл {csv_path} не содержит всех ожидаемых столбцов {expected_columns}")
        continue
    
    # Создание нового DataFrame с требуемыми полями
    new_df = pd.DataFrame({
        "img_name": df["image_path"].apply(lambda x: os.path.basename(x)),  # Извлекаем имя файла
        "label": df["text"].fillna(""),  # Текст бокса, заменяем NaN на пустую строку
        "bbox": df["bbox"].fillna(""),   # Координаты бокса, заменяем NaN на пустую строку
        "img_path": df["image_path"]     # Полный путь к файлу
    })
    
    # Сохранение DataFrame в словарь
    dataframes[df_name] = new_df
    
    # Вывод информации о DataFrame
    print(f"\nСоздан DataFrame для {csv_path}:")
    print(f"Имя DataFrame: {df_name}")
    print(f"Количество строк: {len(new_df)}")
    print("Первые 2 строки:")
    print(new_df.head(2))
    print()

# Проверка, были ли созданы DataFrame'ы
if dataframes:
    print("Созданные DataFrame'ы:", list(dataframes.keys()))
else:
    print("Ни один DataFrame не был создан из-за отсутствия файлов или ошибок в данных.")

# Сохраняем train_df и test_df в указанные CSV-файлы
if 'train_df' in dataframes:
    dataframes['train_df'].to_csv('d2_1_train.csv', index=False, encoding='utf-8-sig')
    print("✅ train_df сохранён в d2_1_train.csv")

if 'test_df' in dataframes:
    dataframes['test_df'].to_csv('d2_1_test.csv', index=False, encoding='utf-8-sig')
    print("✅ test_df сохранён в d2_1_test.csv")
    


Создан DataFrame для image_text_pairs2_1.csv:
Имя DataFrame: val_df
Количество строк: 34623
Первые 2 строки:
   img_name   label                                               bbox  \
0  2638.jpg  Своими  891.09,1779.73,276.26999999999987,94.289999999...   
1  2638.jpg      он  2084.49,1889.88,82.02000000000044,51.879999999...   

                     img_path  
0  dataset2_1/images/2638.jpg  
1  dataset2_1/images/2638.jpg  


Создан DataFrame для annotations_test.csv:
Имя DataFrame: test_df
Количество строк: 34670
Первые 2 строки:
   img_name    label                                               bbox  \
0  2013.jpg  узнавая  3028.14,430.23,328.6800000000003,62.0799999999...   
1  2013.jpg   больше  2748.04,426.76,268.6300000000001,58.6000000000...   

                     img_path  
0  dataset2_1/images/2013.jpg  
1  dataset2_1/images/2013.jpg  


Создан DataFrame для annotations_train.csv:
Имя DataFrame: train_df
Количество строк: 335962
Первые 2 строки:
  img_name   label          

In [ ]:
dataframes['train_df'].head(25)
# dataframes['test_df']
# dataframes['val_df']

,img_name,label,bbox,img_path
0,0_0.jpg,"прил.,","1301.01,1430.32,163.73000000000002,68.93000000...",dataset2_1/images/0_0.jpg
1,0_0.jpg,"п.,","1610.74,1503.95,73.41000000000008,59.099999999...",dataset2_1/images/0_0.jpg
2,0_0.jpg,П.,"1525.52,1478.45,84.80999999999995,75.269999999...",dataset2_1/images/0_0.jpg
3,0_0.jpg,в,"1483.43,1486.12,41.87999999999988,64.490000000...",dataset2_1/images/0_0.jpg
4,0_0.jpg,"ч.,","1381.82,1506.44,75.26999999999998,63.450000000...",dataset2_1/images/0_0.jpg
5,0_0.jpg,ж.,"1193.52,275.78,51.680000000000064,21.540000000...",dataset2_1/images/0_0.jpg
6,0_0.jpg,ед.,"1300.08,1509.35,76.31000000000017,80.450000000...",dataset2_1/images/0_0.jpg
7,0_0.jpg,в,"1248.86,1498.98,41.47000000000003,64.900000000...",dataset2_1/images/0_0.jpg
8,0_0.jpg,"р.,","1144.35,1517.43,80.46000000000004,80.669999999...",dataset2_1/images/0_0.jpg
9,0_0.jpg,сообщали,"1116.47,965.43,251.02999999999997,82.419999999...",dataset2_1/images/0_0.jpg
